# Exercise 1.3: Grid Search for Hyperparameter Optimization

## importing :

In [23]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Train/Test split
from sklearn.model_selection import train_test_split

# ANN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Evaluation
from sklearn.model_selection import cross_val_score
from scikeras.wrappers import KerasClassifier

# Grid Search
from sklearn.model_selection import GridSearchCV

## loading dataset :

In [24]:
dataset = pd.read_csv('Churn_Modelling.csv')

X = dataset.iloc[:, 3:-1].values
y = dataset.iloc[:, -1].values

## Encoding Categorical Data :

In [25]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Geography
labelencoder_X_1 = LabelEncoder()
X[:, 1] = labelencoder_X_1.fit_transform(X[:, 1])

# Gender
labelencoder_X_2 = LabelEncoder()
X[:, 2] = labelencoder_X_2.fit_transform(X[:, 2])

# One-hot encoding Geography
ct = ColumnTransformer(
    [('Geography', OneHotEncoder(), [1])],
    remainder='passthrough'
)

X = ct.fit_transform(X)

# Avoid dummy variable trap
X = X[:, 1:]

## Training Testing Spliting :

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

## Feature Scaling :

In [27]:
sc = StandardScaler()

X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## Building ANN :

In [28]:
def build_classifier():
    classifier = Sequential()
    classifier.add(Dense(units=6, activation='relu', input_dim=11))
    classifier.add(Dense(units=6, activation='relu'))
    classifier.add(Dense(units=1, activation='sigmoid'))
    classifier.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return classifier

## K-Fold Cross Validation :

In [31]:
from sklearn.model_selection import KFold
import numpy as np

kfold = KFold(n_splits=10, shuffle=True, random_state=0)

accuracies = []

for train_idx, val_idx in kfold.split(X_train):
    model = build_classifier()
    
    model.fit(
        X_train[train_idx],
        y_train[train_idx],
        epochs=100,
        batch_size=10,
        verbose=0
    )
    
    loss, acc = model.evaluate(
        X_train[val_idx],
        y_train[val_idx],
        verbose=0
    )
    
    accuracies.append(acc)

accuracies = np.array(accuracies)

print("Mean:", accuracies.mean())
print("Std:", accuracies.std())

# used manual k fold

/home/athena/anaconda3/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1773866802.658530   13029 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Mean: 0.8572499990463257
Std: 0.012891377781942562


In [33]:
print("Mean:", accuracies.mean())
print("Std:", accuracies.std())

Mean: 0.8572499990463257
Std: 0.012891377781942562


The mean accuracy represents the average performance of the ANN

The standard deviation shows how stable the model is:

Small std → stable model

Large std → unstable model


our model is stable